# Descriptive and Diagnostic Analytics of Online Shopping Customer Behavior

This notebook supports the ITBAN 3 documentation and ITBAN 4 implementation. It analyzes the Online Shoppers Purchasing Intention dataset using descriptive analytics and diagnostic analytics only.

## Member 1 - Introduction and Dataset Loading

In this section, we import the libraries and load the dataset. We inspect the first rows, dataset shape, and data types.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind

sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/online_shoppers_intention.csv")
print(df.head())
print("Shape:", df.shape)
print(df.info())

## Member 2 - Data Cleaning and Preparation

We check missing values and duplicates, then remove duplicates. Revenue and Weekend are converted to integer fields for analysis.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

clean_df = df.drop_duplicates().copy()
clean_df["Revenue_Int"] = clean_df["Revenue"].astype(int)
clean_df["Weekend_Int"] = clean_df["Weekend"].astype(int)
clean_df["Purchase_Status"] = clean_df["Revenue_Int"].map({0: "No Purchase", 1: "Purchase"})
clean_df["Weekend_Label"] = clean_df["Weekend_Int"].map({0: "Weekday", 1: "Weekend"})

month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
clean_df["Month"] = pd.Categorical(clean_df["Month"], categories=month_order, ordered=True)
clean_df["ProductRelated_Level"] = pd.qcut(
    clean_df["ProductRelated"].rank(method="first"),
    q=4,
    labels=["Q1 Low", "Q2 Lower-Mid", "Q3 Upper-Mid", "Q4 High"]
)
clean_df["PageValue_Group"] = np.where(clean_df["PageValues"] > 0, "With Page Value", "Zero Page Value")

clean_df.to_csv("../data/cleaned_online_shoppers_intention.csv", index=False)
print("Cleaned shape:", clean_df.shape)
clean_df.head()

## Member 3 - Descriptive Analytics

This is the “what happened” part of the project. We summarize purchase outcomes, visitor types, monthly activity, and conversion rates.

In [ ]:
print(clean_df.describe())

purchase_rate = clean_df["Revenue_Int"].mean() * 100
print(f"Overall purchase rate: {purchase_rate:.2f}%")

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=clean_df, x="Purchase_Status", order=["No Purchase", "Purchase"], color="#4C72B0")
for c in ax.containers:
    ax.bar_label(c, fmt="%d")
plt.title("Figure 1. Purchase vs No Purchase Sessions")
plt.xlabel("Session Outcome")
plt.ylabel("Number of Sessions")
plt.show()

plt.figure(figsize=(7, 4))
ax = sns.countplot(data=clean_df, x="VisitorType", order=clean_df["VisitorType"].value_counts().index, color="#4C72B0")
for c in ax.containers:
    ax.bar_label(c, fmt="%d")
plt.title("Figure 2. Visitor Type Distribution")
plt.xlabel("Visitor Type")
plt.ylabel("Number of Sessions")
plt.show()

plt.figure(figsize=(8, 4))
ax = sns.countplot(data=clean_df, x="Month", order=month_order, color="#4C72B0")
for c in ax.containers:
    ax.bar_label(c, fmt="%d", fontsize=7)
plt.title("Figure 3. Number of Sessions by Month")
plt.xlabel("Month")
plt.ylabel("Number of Sessions")
plt.show()

In [ ]:
month_summary = clean_df.groupby("Month", observed=True).agg(
    sessions=("Revenue_Int", "size"),
    purchases=("Revenue_Int", "sum"),
    purchase_rate=("Revenue_Int", "mean"),
    avg_page_value=("PageValues", "mean"),
    avg_product_pages=("ProductRelated", "mean")
).reset_index()
month_summary["purchase_rate_pct"] = month_summary["purchase_rate"] * 100
print(month_summary)

plt.figure(figsize=(8, 4))
sns.lineplot(data=month_summary, x="Month", y="purchase_rate_pct", marker="o")
plt.title("Figure 4. Purchase Conversion Rate by Month")
plt.xlabel("Month")
plt.ylabel("Purchase Conversion Rate (%)")
plt.show()

## Member 4 - Diagnostic Analytics

This is the “why did it happen” part of the project. We examine correlations, buyer vs. non-buyer behavior, product-page engagement, and outliers.

In [ ]:
cols_for_corr = [
    "Revenue_Int", "PageValues", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "Administrative", "Informational", "SpecialDay", "Weekend_Int"
]

plt.figure(figsize=(10, 8))
sns.heatmap(clean_df[cols_for_corr].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Figure 5. Correlation Heatmap of Key Variables")
plt.show()

revenue_corr = clean_df[cols_for_corr].corr()["Revenue_Int"].drop("Revenue_Int").sort_values(ascending=False)
print(revenue_corr)

In [ ]:
buyer_comparison = clean_df.groupby("Purchase_Status")[[
    "ProductRelated", "ProductRelated_Duration", "PageValues", "BounceRates", "ExitRates"
]].mean().loc[["No Purchase", "Purchase"]]
print(buyer_comparison)

buyer_comparison[["ProductRelated", "PageValues", "BounceRates", "ExitRates"]].plot(kind="bar", figsize=(9, 5))
plt.title("Figure 6. Average Behavior of Buyers vs Non-Buyers")
plt.xlabel("Session Outcome")
plt.ylabel("Average Value")
plt.xticks(rotation=0)
plt.show()

page_value_group = clean_df.groupby("PageValue_Group")["Revenue_Int"].agg(["count", "sum", "mean"]).reset_index()
page_value_group["purchase_rate_pct"] = page_value_group["mean"] * 100
print(page_value_group)

plt.figure(figsize=(7, 4))
ax = sns.barplot(data=page_value_group, x="PageValue_Group", y="purchase_rate_pct", color="#4C72B0")
for c in ax.containers:
    ax.bar_label(c, fmt="%.1f%%")
plt.title("Figure 7. Purchase Rate by Page Value Group")
plt.xlabel("Page Value Group")
plt.ylabel("Purchase Rate (%)")
plt.show()

In [ ]:
# Outlier analysis using IQR
for col in ["ProductRelated_Duration", "ProductRelated", "PageValues", "BounceRates", "ExitRates"]:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = clean_df[(clean_df[col] < lower) | (clean_df[col] > upper)]
    print(col, "outliers:", len(outliers), "upper bound:", round(upper, 4), "max:", round(clean_df[col].max(), 4))

plt.figure(figsize=(8, 4))
sns.boxplot(data=clean_df, x="Purchase_Status", y="ProductRelated_Duration", order=["No Purchase", "Purchase"])
plt.title("Figure 8. Product-Related Duration Outlier Pattern")
plt.xlabel("Session Outcome")
plt.ylabel("Product Related Duration")
plt.show()

## Member 5 - Statistical Test and Conclusion

A chi-square test is used to test whether VisitorType is related to purchase outcome.

In [ ]:
contingency_table = pd.crosstab(clean_df["VisitorType"], clean_df["Revenue_Int"])
print(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))
print("Cramer's V:", cramers_v)

## Final Takeaway

The cleaned dataset contains **12,205 sessions**. The overall purchase rate is **15.63%**, meaning most sessions did not convert into purchases. Diagnostic results show that **PageValues**, product-related activity, and visitor type are useful factors for understanding purchase behavior. Businesses should improve product pages, encourage deeper product browsing, and design visitor-specific strategies to increase purchase conversion.